# Banking Intent Classification with Unsloth & QLoRA

This notebook fine-tunes a Llama-3 8B model for intent classification on the BANKING77 dataset.

**Requirements:**
- Kaggle GPU T4 x2 (Settings > Accelerator > GPU T4 x2)
- Internet enabled (Settings > Internet > On)
- Persistence: Files only

## Step 1: Setup — Clone Repository & Install Dependencies

In [ ]:
import os

REPO_URL  = "https://github.com/tunah72/banking-intent-unsloth.git"
REPO_NAME = "banking-intent-unsloth"

if os.path.exists(REPO_NAME):
    print(f"Repo already cloned. Pulling latest changes...")
    os.system(f"git -C {REPO_NAME} pull")
else:
    os.system(f"git clone {REPO_URL}")

os.chdir(REPO_NAME)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install project dependencies
# peft + bitsandbytes handle QLoRA; no unsloth dependency needed
!pip install -r requirements.txt -q

# Verify critical imports
from transformers import AutoModelForSequenceClassification
from peft import PeftModel, LoraConfig
import bitsandbytes
print(f'transformers: OK')
print(f'peft:         OK')
print(f'bitsandbytes: OK')


## Step 2: Data Preprocessing

Downloads the BANKING77 dataset from HuggingFace and applies Stratified Sampling:
- **Train:** 50 samples/label (~3,826 total — some labels have fewer in source)
- **Validation:** 5 samples/label (~375 total)
- **Test:** 10 samples/label (770 total — exactly 10 per label)

Text normalization (strip + lowercase) is applied to all splits.

In [ ]:
!python scripts/preprocess_data.py

## Step 3: Model Fine-tuning

Fine-tunes `unsloth/llama-3-8b-bnb-4bit` using QLoRA with the following settings:
- LoRA rank: 16, alpha: **32**, dropout: 0.05
- Effective batch size: 8 (batch_size=2 × gradient_accumulation=4)
- Learning rate: 2e-4, warmup_ratio: 0.05, scheduler: linear
- Max sequence length: 128
- Best checkpoint selected by **eval_accuracy** (not loss)

In [ ]:
!bash train.sh

## Step 4: Inference Demo

Loads the trained model and predicts intent labels for a predefined set of sample banking queries.

In [ ]:
!bash inference.sh

## Step 5: Model Evaluation

Evaluates the fine-tuned model on the test set (770 samples) and generates:
- `results/test_predictions.csv` — Full prediction log
- `results/classification_report.txt` — Precision, Recall, F1 per label
- `results/metrics.json` — Raw metrics in JSON format
- `results/confusion_matrix.png` — Confusion matrix heatmap

In [ ]:
!bash evaluate.sh

### Evaluation Results

In [ ]:
# Overall metrics summary
import json

with open("results/metrics.json", "r") as f:
    metrics = json.load(f)

macro = metrics.get("macro avg", {})
accuracy = metrics.get("accuracy", 0)

print("=" * 50)
print(" FINAL EVALUATION METRICS")
print("=" * 50)
print(f"  Overall Accuracy : {accuracy * 100:.2f}%")
print(f"  Macro Precision  : {macro.get('precision', 0) * 100:.2f}%")
print(f"  Macro Recall     : {macro.get('recall', 0) * 100:.2f}%")
print(f"  Macro F1-Score   : {macro.get('f1-score', 0) * 100:.2f}%")
print("=" * 50)

In [ ]:
# Per-class classification report
with open("results/classification_report.txt", "r") as f:
    print(f.read())

In [ ]:
# Confusion matrix heatmap
from IPython.display import Image, display
display(Image(filename="results/confusion_matrix.png", width=900))

## Step 6: Save Results

Copies all outputs to `/kaggle/working/output/` for persistent storage.

After running this cell, click **Save Version** > **Save & Run All (Commit)** to store permanently.

In [ ]:
import shutil
import os

output_dir = "/kaggle/working/output"
os.makedirs(output_dir, exist_ok=True)

# Copy model checkpoint
if os.path.exists("checkpoints/final_best_model"):
    shutil.copytree("checkpoints/final_best_model", f"{output_dir}/final_best_model", dirs_exist_ok=True)
    print("Copied model checkpoint.")
else:
    print("[WARN] Checkpoint not found — did training complete?")

# Copy evaluation results
if os.path.exists("results"):
    shutil.copytree("results", f"{output_dir}/results", dirs_exist_ok=True)
    print("Copied evaluation results.")

# Copy preprocessed data
if os.path.exists("sample_data"):
    shutil.copytree("sample_data", f"{output_dir}/sample_data", dirs_exist_ok=True)
    print("Copied preprocessed data.")

print(f"\nAll outputs saved to {output_dir}")
print("Click Save Version to persist these files on Kaggle.")